# 음성 파일 첨부 재현 — `handler_client` 는 되고 `llm_manager` 는 안 된다

**목적**: 동일한 음성 파일이 경로에 따라 갈리는 것을 대조한다.

```
같은 test_audio.m4a, 같은 llm-handler, 같은 Gemini

  경로 A  handler_client.call([..., Part.from_file(m4a)])
            → mime 정규화 → llm-handler → Gemini      ✅ 성공

  경로 B  llm_request_async(stored_files=[StoredFile])
            → validate_mime_types 화이트리스트        ❌ UnsupportedFileTypeError
                                                        (LLM 에 닿지도 못한다)
```

**밝혀낸 결함은 둘이다.**

1. **정규화 순서 결함** — `validators.SUPPORTED_MIME_TYPES` 는 정규화된 이름(`audio/mp3`)으로
   적혀 있는데, 대조되는 값은 정규화 전 이름(`audio/mpeg`)이다. `mimetypes.guess_type` 이 내놓는
   `audio/mpeg`·`audio/x-wav`·`audio/x-aiff`·`video/quicktime` 이 전부 거짓 거부된다.
2. **화이트리스트 과잉 제한** — `audio/mp4`(`.m4a`)는 Gemini 가 실제로 받아주는데도 목록에 없어
   거부된다. 경로 A 가 그 증거다.

> **개인정보 주의** — `test_audio.m4a` 는 실제 사내 대화 녹음이다. 이 노트북은 전사 **내용을
> 출력하지 않고** 응답 길이와 성공 여부만 찍는다. 노트북 출력이 git 에 저장되기 때문이다.

관련 이슈: KiwoomAM/Kiwoom-Manager#67

> **저장된 출력은 수정 전(pre-fix)에 캡처한 것이다.** 11·18번 셀의 `UnsupportedFileTypeError` 가
> 이 버그의 증거다. 수정이 적용된 지금 다시 실행하면 두 셀 모두 `[예상 밖] 성공했다` 로 바뀌는데,
> 그것이 정상이다. 22번 셀이 수정 후의 계약을 확인한다.

## 설정

> **설치 셀을 두지 않는다.** `llm-manager/.venv` 는 uv 로 만들어져 `pip` 모듈이 없어서,
> `%pip install` 은 `No module named pip` 으로 실패한다. 필요한 패키지
> (`llm_manager`·`file_manager`·`mdb_manager`·`redis_manager`)는 이미 그 venv 에 들어 있다.
> 없다면 노트북이 아니라 셸에서 `uv sync` 로 채운다.

In [2]:
# 커널이 기대한 venv 를 잡았는지 먼저 확인한다 — 다른 인터프리터면 아래가 전부 ImportError 다.
import sys

print(f"  python      = {sys.executable}")
print(f"  version     = {sys.version.split()[0]}")

import file_manager  # noqa: F401
import llm_manager  # noqa: F401

print("  llm_manager / file_manager import OK")

  python      = /home/junyoung_oh/projects/kiwoom/kiwoom-manager/llm-manager/.venv/bin/python
  version     = 3.13.11
  llm_manager / file_manager import OK


In [3]:
import os
import nest_asyncio

from pathlib import Path
from dotenv import load_dotenv

nest_asyncio.apply()

# .env 는 커널이 자동으로 안 읽음 → Kiwoom-Manager/.env 를 명시 로드 (handler URL 전부).
load_dotenv(Path.cwd().parents[2] / ".env", override=True)

os.environ["MONGODB_DATABASE"] = "kw_ai_dev"  # 테스트 전용 DB
TEST_DB = os.environ["MONGODB_DATABASE"]

# handler_client 를 llm_request 보다 먼저 쓰므로 LLM_HANDLER_URL 을 여기서 세운다.
#
#   llm_manager 의 ensure_configured() 가 이 변수를 사내 기본값으로 setdefault 하지만,
#   그것은 llm_request 계열을 처음 부를 때만 돈다. 이 노트북의 경로 A 는 handler_client 를
#   직접 쓰므로 그 부트스트랩을 건너뛰고, handler_client 자체 기본값인 localhost:8100 으로
#   붙으려다 ConnectionRefused 로 죽는다.
#
#   IP 를 노트북에 박지 않으려고 manager 의 기본값을 SSOT 로 끌어 쓴다.
from llm_manager._internal._config import LLM_HANDLER_URL

os.environ.setdefault("LLM_HANDLER_URL", LLM_HANDLER_URL)
os.environ.setdefault("MDB_HANDLER_URL", "http://192.7.9.45:8300")
os.environ.setdefault("FILE_HANDLER_URL", "http://192.7.9.45:8600")
os.environ.setdefault("REDIS_URL", "http://192.7.9.45:8800")

for k in ("MDB_HANDLER_URL", "LLM_HANDLER_URL", "FILE_HANDLER_URL", "REDIS_URL", "MONGODB_DATABASE"):
    print(f"  {k} = {os.environ.get(k)}")

  MDB_HANDLER_URL = http://192.7.9.45:8300
  LLM_HANDLER_URL = http://192.7.9.45:8100
  FILE_HANDLER_URL = http://192.7.9.45:8600
  REDIS_URL = http://192.7.9.45:8800
  MONGODB_DATABASE = kw_ai_dev


In [4]:
# 핸들러가 실제로 살아 있는지 먼저 확인한다 — 죽어 있으면 아래 셀들이 전부 ConnectionRefused 로
# 죽는데, 그 예외는 우리가 보려는 MIME 결함과 무관하다. 여기서 시끄럽게 끊는 편이 낫다.
import httpx

health = httpx.get(os.environ["LLM_HANDLER_URL"] + "/health", timeout=5)
health.raise_for_status()
print(f"  llm-handler {os.environ['LLM_HANDLER_URL']} → {health.status_code} {health.text}")

  llm-handler http://192.7.9.45:8100 → 200 {"status":"ok"}


In [5]:
from pathlib import Path

AUDIO_M4A = Path("test_audio.m4a").resolve()
MODEL = "gemini-2.5-flash"

assert AUDIO_M4A.exists(), f"{AUDIO_M4A} 없음"
print(f"{AUDIO_M4A.name}  {AUDIO_M4A.stat().st_size:,}B")

test_audio.m4a  4,006,244B


---
## 경로 A — `handler_client` (화이트리스트 없음) → 성공한다

`Part.from_file()` 이 확장자와 매직 바이트로 MIME 을 감지하고 `_normalize_mime_type` 을 거친다.
`.m4a` 는 `audio/mp4` 로 감지되어 그대로 Gemini 로 나가고, Gemini 는 이를 받아들인다.
공식 문서의 지원 목록에는 `audio/mp4` 가 없지만 실제로는 동작한다.

In [6]:
from llm_manager._internal.handler_client import Part, call

part = Part.from_file(str(AUDIO_M4A))
print(f"  Part.from_file 이 결정한 mime = {part.mime_type}")

response = call(["이 오디오의 내용을 한 문장으로 요약해줘", part], model=MODEL)

# 전사 내용은 사내 대화라 출력하지 않는다 (위 개인정보 주의 참고).
print(f"  [OK] Gemini 응답 수신 — {len(response.text)}자")

  Part.from_file 이 결정한 mime = audio/mp4
  [OK] Gemini 응답 수신 — 87자


---
## 경로 B — `llm_manager` 의 `stored_files` → 거부된다

`file-manager` 의 `build_metadata` 가 `mimetypes.guess_type` 으로 MIME 을 만들고,
`resolve_stored_files` 가 그 값을 손대지 않은 채 `validate_mime_types` 에 넘긴다.
`audio/mp4` 는 화이트리스트에 없으므로 **LLM 호출 이전에** 예외로 끊긴다.

In [7]:
from file_manager import FileManagerHandler
from llm_manager import StoredFile, llm_request_async, UnsupportedFileTypeError

fm = FileManagerHandler()  # 싱글톤


async def build_stored_file(path: Path, mime_override: str | None = None) -> StoredFile:
    """file-manager 에 등록해 StoredFile 을 만든다.

    mime_override 를 주면 build_metadata 가 감지한 mime 을 덮어쓴다 (우회책 실증용).
    주의: dedup 키는 file_name/file_mtime/file_size 3필드다. 같은 파일을 mime 만 바꿔
    다시 등록하면 캐시가 예전 mime 을 돌려주므로, 우회 실증에는 파일명을 달리해야 한다.
    """
    meta = fm.build_metadata(
        file_name=path.name,
        file_mtime=int(path.stat().st_mtime),
        file_size=path.stat().st_size,
    )
    if mime_override is not None:
        meta["mime_type"] = mime_override
    sf_dict = await fm.register_or_get(bytes_data=path.read_bytes(), metadata=meta)
    return StoredFile(**sf_dict)

In [8]:
sf_m4a = await build_stored_file(AUDIO_M4A)
print(f"  build_metadata 가 감지한 mime = {sf_m4a.mime_type}")

try:
    r, m = await llm_request_async(
        {"contents": "이 오디오의 내용을 한 문장으로 요약해줘", "model": MODEL},
        stored_files=[sf_m4a],
        database=TEST_DB,
    )
    print(f"  [예상 밖] 성공했다 — {len(r.text)}자. 화이트리스트가 이미 고쳐졌는지 확인할 것")
except UnsupportedFileTypeError as e:
    print(f"  [재현] UnsupportedFileTypeError: {e}")
    print("        경로 A 는 같은 파일로 성공했다 → Gemini 가 아니라 매니저가 막은 것이다")

  build_metadata 가 감지한 mime = audio/mp4
  [재현] UnsupportedFileTypeError: 지원하지 않는 파일 타입: mime_type=audio/mp4, file_name=test_audio.m4a
        경로 A 는 같은 파일로 성공했다 → Gemini 가 아니라 매니저가 막은 것이다


---
## 매니저의 화이트리스트를 해부한다 — 네트워크 없이

> **아래 `판정` 열의 주어는 Gemini 가 아니라 `llm_manager` 다.** 이 표는 `guess_type` 이 준 MIME 을
> `validators.SUPPORTED_MIME_TYPES` 에 대조할 뿐이다. 경로 A 가 보여주듯 `handler_client` 는
> 화이트리스트 자체를 갖지 않고 Gemini 에게 판단을 위임한다. 그래서 여기서 `거부` 로 찍히는
> 타입도 Gemini 는 받아줄 수 있다.

두 종류의 거부가 섞여 있으니 구분해서 읽는다.

- **거짓 거부** — 화이트리스트는 정규화 **후** 이름(`audio/mp3`)으로 적혀 있는데 대조되는 값은
  정규화 **전** 이름(`audio/mpeg`)이다. 정규화만 먼저 했으면 통과했을 값이다.
- **목록 누락** — 정규화해도 목록에 없다. `audio/mp4` 가 여기 해당하는데, 경로 A 에서 실제로
  Gemini 를 통과했으므로 이것은 화이트리스트가 지나치게 좁다는 뜻이다.

In [9]:
import mimetypes

from llm_manager._internal.file_processing.validators import SUPPORTED_MIME_TYPES
from llm_manager._internal.handler_client._config import get_settings

NORMALIZE = get_settings().gemini_mime_normalize

print(f"{'확장자':<8}{'guess_type':<20}{'정규화 후':<20}{'llm_manager 의 판정'}")
print("-" * 84)
for ext in [".mp3", ".wav", ".aiff", ".mov", ".aac", ".flac", ".ogg", ".m4a"]:
    raw = mimetypes.guess_type("a" + ext)[0] or "application/octet-stream"
    norm = NORMALIZE.get(raw, raw)
    if raw in SUPPORTED_MIME_TYPES:
        verdict = "통과시킨다"
    elif norm in SUPPORTED_MIME_TYPES:
        verdict = "거짓 거부 — 정규화만 먼저 했으면 통과했을 값"
    else:
        verdict = "거부 — 목록에 없다 (Gemini 의 판단이 아니다)"
    print(f"{ext:<8}{raw:<20}{norm:<20}{verdict}")

print(f"\n  화이트리스트의 오디오 항목: {sorted(m for m in SUPPORTED_MIME_TYPES if m.startswith('audio/'))}")
print(f"  audio/mp4 등재 여부: {'audio/mp4' in SUPPORTED_MIME_TYPES}  (그러나 경로 A 는 성공했다)")

확장자     guess_type          정규화 후               llm_manager 의 판정
------------------------------------------------------------------------------------
.mp3    audio/mpeg          audio/mp3           거짓 거부 — 정규화만 먼저 했으면 통과했을 값
.wav    audio/x-wav         audio/wav           거짓 거부 — 정규화만 먼저 했으면 통과했을 값
.aiff   audio/x-aiff        audio/aiff          거짓 거부 — 정규화만 먼저 했으면 통과했을 값
.mov    video/quicktime     video/mov           거짓 거부 — 정규화만 먼저 했으면 통과했을 값
.aac    audio/aac           audio/aac           통과시킨다
.flac   audio/flac          audio/flac          통과시킨다
.ogg    audio/ogg           audio/ogg           통과시킨다
.m4a    audio/mp4           audio/mp4           거부 — 목록에 없다 (Gemini 의 판단이 아니다)

  화이트리스트의 오디오 항목: ['audio/aac', 'audio/aiff', 'audio/flac', 'audio/mp3', 'audio/ogg', 'audio/wav']
  audio/mp4 등재 여부: False  (그러나 경로 A 는 성공했다)


> `.flac` 과 `.ogg` 의 `통과` 는 안정적이지 않다. 파이썬 내장 매핑에는 두 확장자가 없어 `None` 이며,
> 위 값은 OS 의 `/etc/mime.types` 가 공급한 것이다. 그 항목이 없는 호스트에서는
> `application/octet-stream` 으로 떨어져 `.mp3` 와 똑같이 거부된다. 아래 셀로 확인한다.

In [10]:
import mimetypes

builtin = getattr(mimetypes, "_types_map_default", None)
if builtin is None:
    print("  이 파이썬에는 내장 맵이 노출돼 있지 않다 — 건너뛴다")
else:
    print(f"{'확장자':<8}{'내장 맵':<20}{'이 호스트':<20}")
    print("-" * 48)
    for ext in [".mp3", ".wav", ".aiff", ".mov", ".aac", ".flac", ".ogg", ".m4a"]:
        print(f"{ext:<8}{str(builtin.get(ext)):<20}{str(mimetypes.guess_type('a' + ext)[0]):<20}")
    print("\n  내장 맵이 None 인 확장자는 호스트의 /etc/mime.types 에 의존한다")

확장자     내장 맵                이 호스트               
------------------------------------------------
.mp3    audio/mpeg          audio/mpeg          
.wav    audio/x-wav         audio/x-wav         
.aiff   audio/x-aiff        audio/x-aiff        
.mov    video/quicktime     video/quicktime     
.aac    audio/aac           audio/aac           
.flac   None                audio/flac          
.ogg    None                audio/ogg           
.m4a    None                audio/mp4           

  내장 맵이 None 인 확장자는 호스트의 /etc/mime.types 에 의존한다


---
## 정규화 순서 결함 — 살아 있는 재현 (합성 WAV)

`.wav` 는 `audio/x-wav` 로 감지되어 거짓 거부되는 대표 사례다. 음성 원본을 준비할 필요 없이
표준 라이브러리 `wave` 만으로 만들 수 있다. 전사 내용은 무의미하지만, 우리가 보려는 것은
**MIME 게이트를 통과하는가** 하나뿐이다.

우회책 실증에는 dedup 충돌을 피하려고 파일명을 달리한 사본을 쓴다.

In [11]:
import math
import struct
import wave


def write_tone(path: Path, seconds: float = 1.0, rate: int = 16000, freq: float = 440.0) -> Path:
    """440Hz 사인파 모노 WAV 생성 (멱등 — 같은 내용으로 덮어쓴다)."""
    frames = b"".join(
        struct.pack("<h", int(3000 * math.sin(2 * math.pi * freq * t / rate))) for t in range(int(rate * seconds))
    )
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes(frames)
    return path


WAV_DEFAULT = write_tone(Path("repro_tone_default.wav").resolve())
WAV_OVERRIDE = write_tone(Path("repro_tone_override.wav").resolve())
print(f"  생성: {WAV_DEFAULT.name}, {WAV_OVERRIDE.name}  ({WAV_DEFAULT.stat().st_size:,}B)")

  생성: repro_tone_default.wav, repro_tone_override.wav  (32,044B)


In [12]:
# (1) 감지된 mime 그대로 → audio/x-wav → 거짓 거부
sf_bad = await build_stored_file(WAV_DEFAULT)
print(f"  감지된 mime = {sf_bad.mime_type}")

try:
    r, m = await llm_request_async(
        {"contents": "이 소리를 묘사해줘", "model": MODEL},
        stored_files=[sf_bad],
        database=TEST_DB,
    )
    print(f"  [예상 밖] 성공했다 — 결함이 이미 고쳐졌는지 확인할 것")
except UnsupportedFileTypeError as e:
    print(f"  [재현] UnsupportedFileTypeError: {e}")

  감지된 mime = audio/x-wav
  [재현] UnsupportedFileTypeError: 지원하지 않는 파일 타입: mime_type=audio/x-wav, file_name=repro_tone_default.wav


### 우회 — 등록 전에 MIME 을 표준형으로 덮어쓴다

`build_metadata` 가 돌려주는 것은 평범한 dict 이므로 `register_or_get` 전에 고칠 수 있다.
이 셀이 성공하면 **llm-handler 와 Gemini 는 처음부터 이 오디오를 받아줄 수 있었고,
매니저의 검증 순서만이 문제였다**는 것이 증명된다.

In [15]:
# (2) 등록 전에 mime 을 정규화된 이름으로 덮어쓴다 → audio/wav → 통과
# service_name 등록
os.environ.setdefault("SERVICE_NAME", "test")

sf_ok = await build_stored_file(WAV_OVERRIDE, mime_override="audio/wav")
print(f"  덮어쓴 mime = {sf_ok.mime_type}")

r, m = await llm_request_async(
    {"contents": "이 소리를 묘사해줘", "model": MODEL},
    stored_files=[sf_ok],
    database=TEST_DB,
)
print(f"  [OK] 통과 — {len(r.text)}자, source={m.source}")
print("       같은 바이트인데 mime 이름만 바꿔 통과했다 → 콘텐츠가 아니라 라벨이 문제였다")

  덮어쓴 mime = audio/wav
  [OK] 통과 — 257자, source=live
       같은 바이트인데 mime 이름만 바꿔 통과했다 → 콘텐츠가 아니라 라벨이 문제였다


---
## 고친 자리 — 검증 **전에** 정규화하고, 화이트리스트를 넓힌다

두 결함을 각각 이렇게 막았다.

- **정규화 순서** — `validators.normalize_mime_types()` 를 새로 두고, `resolve_stored_files` 가
  `validate_mime_types()` 를 부르기 **전에** 호출한다. 호출부가 같은 placeholder 를 이후 fetch·업로드에도
  쓰므로 사본이 아니라 **제자리에서** 고치고, 그래서 반환값이 없다(`-> None`).
- **화이트리스트 과잉 제한** — `SUPPORTED_MIME_TYPES` 에 `audio/mp4` 를 더했다. 경로 A 가 Gemini 의
  실제 수용을 증명한다.

핵심은 기본값이다. `gemini_mime_normalize` 는 전체 mime 표가 아니라 **별칭 7개짜리 교정표**다.
그래서 맵에 없는 값은 원본을 그대로 둔다(`get(m, m)`). 기본값을 빈 문자열로 두면 이미 표준형인
`application/pdf`·`image/png`·`audio/aac` 가 전부 파괴된다. 아래 셀이 그 계약을 확인한다.

In [ ]:
import mimetypes

from llm_manager._internal.file_processing.validators import (
    SUPPORTED_MIME_TYPES,
    normalize_mime_types,
    validate_mime_types,
)
from llm_manager.models import FileInput

exts = [".mp3", ".wav", ".aiff", ".mov", ".aac", ".flac", ".m4a", ".png", ".pdf", ".txt"]
probe = [
    FileInput(
        file_name=f"a{e}",
        file_size=1,
        file_mtime=0,
        mime_type=mimetypes.guess_type("a" + e)[0] or "application/octet-stream",
        extension=e,
        data=b"",
    )
    for e in exts
]
before = [f.mime_type for f in probe]

normalize_mime_types(probe)   # 제자리 변형 — 반환값 없음

print(f"{'확장자':<8}{'정규화 전':<20}{'정규화 후':<20}{'화이트리스트'}")
print("-" * 70)
for e, b, fi in zip(exts, before, probe, strict=True):
    ok = "통과" if fi.mime_type in SUPPORTED_MIME_TYPES else "거부"
    print(f"{e:<8}{b:<20}{fi.mime_type:<20}{ok}")

validate_mime_types(probe)   # 예외 없이 통과해야 한다
print("\n  [OK] 10종 전부 통과 — 거짓 거부 4종이 해소되고 audio/mp4 도 살아났다")
print("  [OK] pdf·png·txt 는 정규화 맵에 없지만 원본이 보존됐다 (get(m, m) 의 효과)")

---
## 정리

| 확인한 것 | 결과 |
|---|---|
| 경로 A — `handler_client` + `.m4a` | 성공한다. Gemini 는 `audio/mp4` 를 받아들인다 |
| 경로 B — `stored_files` + `.m4a` | 거부된다. 화이트리스트에 `audio/mp4` 가 없다 |
| 경로 B — `stored_files` + `.wav` | 거부된다. `audio/x-wav` 가 정규화 전에 대조된다 |
| 경로 B — `.wav` + mime 덮어쓰기 | 성공한다. 콘텐츠가 아니라 라벨이 문제였다 |

**결론**: llm-handler 와 Gemini 는 오디오를 지원한다. `llm_manager` 의 `stored_files` 경로만이
검증 순서와 좁은 화이트리스트 때문에 오디오를 잃는다.

`verify_all.ipynb` 17번 절은 `.m4a` 의 거부를 **정상 동작으로 고정**하고 있다. 결함을 사양으로
굳혀 놓은 셈이므로, 화이트리스트를 고칠 때 그 테스트도 함께 바로잡아야 한다.

### 뒷정리 (선택)

합성 WAV 는 재현용 임시 산출물이다. 커밋에 섞이지 않게 지운다.

In [18]:
for p in (WAV_DEFAULT, WAV_OVERRIDE):
    p.unlink(missing_ok=True)
    print(f"  삭제: {p.name}")

  삭제: repro_tone_default.wav
  삭제: repro_tone_override.wav
